# Experiment: Artifact Registry Lineage Checks

Objective:
- Construct a lineage chain
- Compute deterministic IDs
- Validate immutability properties

Done means:
- All assertions pass
- Printed lineage is stable and deterministic


In [ ]:
from __future__ import annotations

import json
import hashlib
from datetime import datetime, timezone

now = datetime(2026, 2, 6, 0, 0, 0, tzinfo=timezone.utc)
TENANT_ID = "tenant_demo"


## Deterministic ID helper


In [ ]:
def deterministic_id(*parts: str) -> str:
    raw = ":".join(parts).encode("utf-8")
    return hashlib.sha256(raw).hexdigest()[:24]


def iso(ts: datetime) -> str:
    return ts.astimezone(timezone.utc).isoformat().replace("+00:00", "Z")


## Build lineage chain


In [ ]:
root = {
    "artifactId": deterministic_id(TENANT_ID, "root", "v1"),
    "type": "ROOT",
    "createdAt": iso(now),
    "parents": []
}

child = {
    "artifactId": deterministic_id(TENANT_ID, root["artifactId"], "v1"),
    "type": "DERIVED",
    "createdAt": iso(now),
    "parents": [root["artifactId"]]
}

grandchild = {
    "artifactId": deterministic_id(TENANT_ID, child["artifactId"], "v1"),
    "type": "DERIVED",
    "createdAt": iso(now),
    "parents": [child["artifactId"]]
}

lineage = [root, child, grandchild]
print(json.dumps(lineage, indent=2))


## Immutability checks


In [ ]:
# Each artifactId must be unique
ids = [a["artifactId"] for a in lineage]
assert len(ids) == len(set(ids))

# Each derived artifact must reference an existing parent
known = set(ids)
for a in lineage[1:]:
    for parent in a["parents"]:
        assert parent in known

# Deterministic: recompute child id and compare
expected_child = deterministic_id(TENANT_ID, root["artifactId"], "v1")
assert child["artifactId"] == expected_child
